# Notebook uniqid tb Custom Table of Contents (Appendix)

In [102]:
toc_generator_code = r"""
import hashlib
import os
from google.colab import _message
import IPython

# --- Helper functions ---

def _get_notebook_cells():
    '''Fetches the notebook\'s cell data from Colab\'s internal API.'''
    resp = _message.blocking_request('get_ipynb')
    if not resp or 'ipynb' not in resp:
        print("❌ Failed to get notebook structure. Please run a cell once and try again.")
        return None
    return resp['ipynb'].get('cells', [])

def _extract_heading_preview(source, cell_type, limit):
    '''Extracts and formats a heading preview from cell source.'''
    if isinstance(source, list):
        text = "".join(source)
    elif isinstance(source, str):
        text = source
    else:
        text = ""

    text = text.strip().replace('\n', ' ') # Corrected to use raw string, so \n is now just \n

    if cell_type == 'markdown' and text.startswith('#'):
        cleaned_heading = text.lstrip('#').strip()
        if not cleaned_heading:
            return "(Empty heading)"
        return f"{cleaned_heading[:limit]}..." if len(cleaned_heading) > limit else cleaned_heading
    else:
        return "(No heading)"

def _generate_table_data(notebook_cells, stored_limit):
    '''Generates Markdown table lines and plain overview lines from notebook cells.'''
    md_table = [
        "### 📑 Quick Navigation (All Cells)",
        "Click each icon to jump directly to that cell.",
        "| Type | Index | Cell ID | Heading |",
        "| :--- | :--- | :--- | :--- |"
    ]

    plain_overview = [
        "Index | Type | Cell ID          | Heading",
        "------|------|------------------|----------------"
    ]

    for idx, cell in enumerate(notebook_cells):
        metadata = cell.get('metadata', {})
        cell_id = metadata.get('colab', {}).get('id') or cell.get('id') or metadata.get('id')
        cell_type = cell.get('cell_type', 'unknown')

        preview = _extract_heading_preview(cell.get('source', ''), cell_type, stored_limit)
        icon = "📝" if cell_type == 'markdown' else "💻"

        is_virtual = False
        if not cell_id:
            raw_data = str(cell.get('source', '')) + str(idx)
            cell_id = "tmp_" + hashlib.md5(raw_data.encode('utf-8')).hexdigest()[:8]
            is_virtual = True

        # Modify md_link to use the index number as link text
        md_link = f"[{idx:03d}](#scrollTo={cell_id})"

        if not is_virtual:
            md_table.append(f"| {icon} | {md_link} | {cell_id} | {preview} |")
            plain_overview.append(f"{idx:03d}   | {icon}  | {cell_id:<16} | {preview}")
        else:
            display_id = f"⚠️ {cell_id}"
            md_table.append(f"| {icon} | {md_link} | {display_id} | {preview} |")
            plain_overview.append(f"{idx:03d}   | {icon}  | {display_id:<14} | {preview}")

    md_table.append("---")
    return md_table, plain_overview

def _save_toc_backup(md_table_content):
    '''Saves the Markdown table content to 'notebook_toc_backup.md'.'''
    backup_content = (
        "# 📑 Notebook TOC Auto-Backup\n\n"
        "## 🔗 Anchor Link View (Copy & Paste to Markdown Cell)\n"
        + "\n".join(md_table_content) + "\n\n"
    )
    with open('notebook_toc_backup.md', 'w', encoding='utf-8') as f:
        f.write(backup_content)

def _print_console_output(md_table, plain_overview, stored_limit):
    '''Prints the Markdown table, plain overview, and environment status to the console.'''
    print("======================= Paste into Md cell: Anchor link ===================")
    for line in md_table:
        print(line)
    print("===========================================================================")

    print("======================= [👀 Plain Text Overview] ==========================")
    for line in plain_overview:
        print(line)
    print("===========================================================================")

    print("======================= [👀 Current Environment Status] ===================")
    print(f"📂 Current Directory : {os.getcwd()}")
    print(f"💾 TOC Saved File   : notebook_toc_backup.md (Markdown format)")
    print(f"⚙️ Stored Max Length: {stored_limit} chars (Configured)")
    print("===========================================================================")

# --- Main function ---
def generate_stable_markdown_toc():
    '''
    [Advanced Reference Version] A definitive code focused on development efficiency,
    integrating %store (persistent settings), %save (automatic file backup),
    and %alias (3-character invocation) to generate 100% reliably functioning copy-paste Markdown.
    '''

    ip = IPython.get_ipython()
    stored_limit = ip.db.get('colab_toc_max_len', 30)

    notebook_cells = _get_notebook_cells()
    if notebook_cells is None:
        return

    md_table, plain_overview = _generate_table_data(notebook_cells, stored_limit)

    _save_toc_backup(md_table)

    _print_console_output(md_table, plain_overview, stored_limit)

def set_toc_len(new_len):
    ip = IPython.get_ipython()
    ip.db['colab_toc_max_len'] = int(new_len)
    print(f"✨ Display length of TOC set to {new_len} characters and saved persistently. Will apply from next execution.")
"""

# Write the code to a file
file_name = "toc_generator.py"
with open(file_name, "w", encoding="utf-8") as f:
    f.write(toc_generator_code)

print(f"Code successfully saved to {file_name}")

Code successfully saved to toc_generator.py


### Using the Modularized Code

Now that the code is saved in `toc_generator.py`, you can import it into your current notebook or any other Python environment. You can then call the `generate_stable_markdown_toc()` and `set_toc_len()` functions as needed.

In [ ]:
import importlib
import toc_generator

# It's good practice to reload the module if you've made changes to toc_generator.py
importlib.reload(toc_generator)

# Set the TOC display length to, for example, 70 characters
toc_generator.set_toc_len(70)

# Generate and print the Table of Contents
toc_generator.generate_stable_markdown_toc()

### How to Use `toc_generator` in Another Colab Notebook

To use the `toc_generator.py` module in another Colab notebook, follow these steps:

1.  **Upload the `toc_generator.py` file**: Open a new Colab notebook, click the file icon (folder mark) on the left. Then, click the upload icon (up arrow) and select the `toc_generator.py` file saved on your computer to upload it.

2.  **Import the module**: Write the following code in a notebook cell to import the module:

    ```python
    import toc_generator
    ```

3.  **Set TOC length (optional)**: If necessary, set the preview length for the generated Table of Contents. This setting will persist across Colab sessions once set.

    ```python
    toc_generator.set_toc_len(70) # Example: set to 70 characters
    ```

4.  **Generate the TOC**: Call the following function to generate the Table of Contents and print it to the console. The output Markdown can be copied and pasted into a Markdown cell in your notebook.

    ```python
    toc_generator.generate_stable_markdown_toc()
    ```

With these steps, you can use the `toc_generator` module to generate a Table of Contents in any other notebook.

## Resources

-   [Google Colaboratory](https://colab.research.google.com/)
-   [GitHub Markdown Guide](https://guides.github.com/features/mastering-markdown/)
-   [Python Official Documentation](https://docs.python.org/3/)

# Colab Notebook Table of Contents Generator

## Why This Tool?

In Google Colaboratory, cell IDs (the unique identifiers for each code or text cell) can change when you add, delete, or reorder cells. This often leads to broken anchor links in your Table of Contents, making it difficult to navigate within your notebook. This tool was developed to solve this problem by generating stable, persistent anchor links that continue to work even after modifications to your notebook structure, ensuring a smooth and reliable navigation experience.

## Overview

This is a Python module designed to generate a stable Markdown-formatted Table of Contents (TOC) for Google Colaboratory notebooks. It specifically addresses the dynamic nature of Colab environments where cell IDs can change, ensuring that the generated anchor links remain functional even after adding or deleting cells.

## Features

*   **Automatic TOC Generation**: Creates a Table of Contents from both Markdown and code cells within a Google Colab notebook.
*   **Stable Anchor Links**: Generates robust anchor links that maintain functionality even when cells are added or removed in the Colab environment.
*   **Customizable Preview Length**: Allows users to customize the length of heading previews displayed in the TOC.
*   **Backup Functionality**: Saves the generated TOC in Markdown format to a `notebook_toc_backup.md` file.
*   **Modular Design**: Can be easily imported and used as a module in any other Colab notebook.

## Installation & Setup

1.  **Download `toc_generator.py`**: Download the `toc_generator.py` file to your local machine.
2.  **Upload to Colab**: In your Google Colab notebook, click the folder icon (Files) on the left sidebar, then click the upload icon (up arrow). Select and upload the `toc_generator.py` file to your Colab runtime.

## Usage

Once the `toc_generator.py` file is uploaded to your Colab environment, you can use it in any notebook:

1.  **Import the module**:

    ```python
    import toc_generator
    ```

2.  **Set TOC display length (optional)**:

    You can set the maximum length of the heading preview in the TOC. This setting will be persistently stored across Colab sessions.

    ```python
    toc_generator.set_toc_len(70) # Example: set to 70 characters
    ```

3.  **Generate the Table of Contents**:

    Call the main function to generate and print the TOC to the console. The output will include a Markdown table ready to be copied and pasted into a Markdown cell in your notebook.

    ```python
    toc_generator.generate_stable_markdown_toc()
    ```

## Example Output

When `toc_generator.generate_stable_markdown_toc()` is executed, it will print output similar to the following in your console:

```
======================= Paste into Md cell: Anchor link ===================
### 📑 Quick Navigation (All Cells)
Click each icon to jump directly to that cell.
| Type | Index | Cell ID | Heading |
| :--- | :--- | :--- | :--- |
| 📝 | [000](#scrollTo=KGkcDF5ZFOo7) | KGkcDF5ZFOo7 | Notebook uniqid tb Custom Table of Contents (Appendix) |
| 💻 | [001](#scrollTo=c10ad51e) | c10ad51e | (No heading) |
| ... (truncated for brevity) ... |
---
===========================================================================
======================= [👀 Plain Text Overview] ==========================
Index | Type | Cell ID          | Heading
------|------|------------------|----------------
000   | 📝  | KGkcDF5ZFOo7     | Notebook uniqid tb Custom Table of Contents (Appendix)
001   | 💻  | c10ad51e         | (No heading)
| ... (truncated for brevity) ... |
===========================================================================
======================= [👀 Current Environment Status] ===================
📂 Current Directory : /content
💾 TOC Saved File   : notebook_toc_backup.md (Markdown format)
⚙️ Stored Max Length: 70 chars (Configured)
===========================================================================
```

*(Note: The actual output will contain all your notebook cells and their respective information.)*

## Development Notes

This project's `toc_generator.py` source code is embedded within its own Colab notebook and generated dynamically. This approach simplifies development and ensures that the module's definition is always available within the working notebook, facilitating iterative changes and testing.

## Contributing

Contributions are welcome! If you have suggestions for improvements, bug reports, or new features, please open an issue or submit a pull request on GitHub.

## License

This project is licensed under the MIT License - see the [LICENSE](LICENSE) file for details.

## Resources

-   [Google Colaboratory](https://colab.research.google.com/)
-   [GitHub Markdown Guide](https://guides.github.com/features/mastering-markdown/)
-   [Python Official Documentation](https://docs.python.org/3/)

In [107]:
readme_content = '''# Cell ID Call: A Colab Table of Contents Generator

## Why This Tool?

In Google Colaboratory, cell IDs (the unique identifiers for each code or text cell) can change when you add, delete, or reorder cells. This often leads to broken anchor links in your Table of Contents, making it difficult to navigate within your notebook. This tool was developed to solve this problem by generating stable, persistent anchor links that continue to work even after modifications to your notebook structure, ensuring a smooth and reliable navigation experience. **Furthermore, a key objective of this tool is to present cell IDs in a manner that makes them immediately identifiable and understandable to human users, even when explained by an AI.**

## Overview

Cell ID Call is a Python module designed to generate a stable Markdown-formatted Table of Contents (TOC) for Google Colaboratory notebooks. It specifically addresses the dynamic nature of Colab environments where cell IDs can change, ensuring that the generated anchor links remain functional even after adding or deleting cells.

## Features

*   **Automatic TOC Generation**: Creates a Table of Contents from both Markdown and code cells within a Google Colab notebook.
*   **Stable Anchor Links**: Generates robust anchor links that maintain functionality even when cells are added or removed in the Colab environment.
*   **Customizable Preview Length**: Allows users to customize the length of heading previews displayed in the TOC.
*   **Backup Functionality**: Saves the generated TOC in Markdown format to a `notebook_toc_backup.md` file.
*   **Modular Design**: Can be easily imported and used as a module in any other Colab notebook.

## Installation & Setup

1.  **Download `toc_generator.py`**: Download the `toc_generator.py` file to your local machine.
2.  **Upload to Colab**: In your Google Colab notebook, click the folder icon (Files) on the left sidebar, then click the upload icon (up arrow). Select and upload the `toc_generator.py` file to your Colab runtime.

## Usage

Once the `toc_generator.py` file is uploaded to your Colab environment, you can use it in any notebook:

1.  **Import the module**: Whereas previously you used the generic 'toc_generator', for this project, you would import 'cell_id_call'.

    ```python
    import cell_id_call as toc_generator
    ```

2.  **Set TOC display length (optional)**:

    You can set the maximum length of the heading preview in the TOC. This setting will be persistently stored across Colab sessions.

    ```python
    toc_generator.set_toc_len(70) # Example: set to 70 characters
    ```

3.  **Generate the Table of Contents**:

    Call the main function to generate and print the TOC to the console. The output will include a Markdown table ready to be copied and pasted into a Markdown cell in your notebook.

    ```python
    toc_generator.generate_stable_markdown_toc()
    ```

## Example Output

When `toc_generator.generate_stable_markdown_toc()` is executed, it will print output similar to the following in your console:

```
======================= Paste into Md cell: Anchor link ===================
### 📑 Quick Navigation (All Cells)
Click each icon to jump directly to that cell.
| Type | Index | Cell ID | Heading |
| :--- | :--- | :--- | :--- |
| 📝 | [000](#scrollTo=KGkcDF5ZFOo7) | KGkcDF5ZFOo7 | Notebook uniqid tb Custom Table of Contents (Appendix) |
| 💻 | [001](#scrollTo=c10ad51e) | c10ad51e | (No heading) |
| ... (truncated for brevity) ... |
---
===========================================================================
======================= [👀 Plain Text Overview] ==========================
Index | Type | Cell ID          | Heading
------|------|------------------|----------------
000   | 📝  | KGkcDF5ZFOo7     | Notebook uniqid tb Custom Table of Contents (Appendix)
001   | 💻  | c10ad51e         | (No heading)
| ... (truncated for brevity) ... |
===========================================================================
======================= [👀 Current Environment Status] ===================
📂 Current Directory : /content
💾 TOC Saved File   : notebook_toc_backup.md (Markdown format)
⚙️ Stored Max Length: 70 chars (Configured)
===========================================================================
```

*(Note: The actual output will contain all your notebook cells and their respective information.)*

## Development Notes

This project's `toc_generator.py` source code is embedded within its own Colab notebook and generated dynamically. This approach simplifies development and ensures that the module's definition is always available within the working notebook, facilitating iterative changes and testing.

## Contributing

Contributions are welcome! If you have suggestions for improvements, bug reports, or new features, please open an issue or submit a pull request on GitHub.

## License

This project is licensed under the MIT License - see the [LICENSE](LICENSE) file for details.

## Resources

-   [Google Colaboratory](https://colab.research.google.com/)
-   [GitHub Markdown Guide](https://guides.github.com/features/mastering-markdown/)
-   [Python Official Documentation](https://docs.python.org/3/)'''

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("README.md has been created. You can download it from the file browser on the left.")

README.md has been created. You can download it from the file browser on the left.
